In [2]:
import numpy as np
import pandas as pd

import pyspark.sql.types as T
import pyspark.sql.functions as F
from yggdrasil.data.etl.spark.init import get_spark_session

In [3]:
NSAMPLES = 10000
NUM_FEATURES = 128

In [4]:
spark = get_spark_session()

25/06/18 15:36:35 WARN Utils: Your hostname, MacBook-Pro-Kasan.local resolves to a loopback address: 127.0.0.1; using 192.168.1.18 instead (on interface en0)
25/06/18 15:36:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/18 15:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
input_matrix = np.random.rand(NSAMPLES, NUM_FEATURES)
df = spark.createDataFrame(
    pd.DataFrame(
        {
            **{
                f"feature_{i}": input_matrix[:, i].tolist()
                for i in range(NUM_FEATURES)
            },
            "target": np.random.randint(2, size=NSAMPLES).tolist(),
        }
    )
)
del input_matrix

In [6]:
select_cols = [f"feature_{i}" for i in range(NUM_FEATURES)]
df = df.withColumn("uid", F.monotonically_increasing_id()).withColumn(
    "sparse_vector",
    F.create_map(
        *list(
            sum(
                [
                    (F.lit(fid).cast("int"), F.col(f"`{fname}`").cast("double"))
                    for fid, fname in enumerate(select_cols)
                ],
                (),
            )
        )
    )
).drop(*select_cols).withColumnRenamed("sparse_vector", "features")
del select_cols

In [7]:
spark.sql("DROP TABLE IF EXISTS demo_processed")
df.select(
    F.col("uid").cast(T.LongType()).alias("uid"),
    F.col("features").cast(T.MapType(T.IntegerType(), T.FloatType())).alias("features"),
    F.col("target").cast(T.ShortType()).alias("target"),
).write.mode("overwrite").saveAsTable("demo_processed")

In [8]:
spark.table("demo_processed").printSchema()

root
 |-- uid: long (nullable = true)
 |-- features: map (nullable = true)
 |    |-- key: integer
 |    |-- value: float (valueContainsNull = true)
 |-- target: short (nullable = true)



In [9]:
spark.stop()